# Logistic Regression — Hyperparameter Tuning

**Objective**: Find the best combination of TF-IDF and Logistic Regression hyperparameters using grid search.

## What We're Tuning

### TF-IDF Parameters
| Parameter | Current | Search Space | What It Controls |
|-----------|---------|-------------|------------------|
| `max_features` | 10,000 | 10k, 30k, 50k | Vocabulary size |
| `ngram_range` | (1,2) | (1,1), (1,2), (1,3) | Unigrams, bigrams, trigrams |
| `sublinear_tf` | False | True, False | Log-normalize term frequency |

### Logistic Regression Parameters
| Parameter | Current | Search Space | What It Controls |
|-----------|---------|-------------|------------------|
| `C` | 1.0 | 0.01, 0.1, 1.0, 10.0 | Regularization strength (lower = more regularization) |

**Total combinations**: 3 × 3 × 2 × 4 = **72 configurations**

We use a **200k sample** with **3-fold cross-validation** to keep training time manageable.

## Table of Contents
1. [Setup & Data Preparation](#1-setup--data-preparation)
2. [Grid Search](#2-grid-search)
3. [Results Analysis](#3-results-analysis)
4. [Best Model Evaluation](#4-best-model-evaluation)
5. [Retrain Best Config on Full Data](#5-retrain-best-config-on-full-data)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import os
import time
from itertools import product

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Settings
pd.set_option('display.max_colwidth', 100)
%matplotlib inline

---
## 1. Setup & Data Preparation

In [ ]:
# Load data
train_df = pd.read_csv(
    'trainingandtestdata/training.1600000.processed.noemoticon.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'id', 'date', 'query', 'user', 'text']
)

test_df = pd.read_csv(
    'trainingandtestdata/testdata.manual.2009.06.14.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'id', 'date', 'query', 'user', 'text']
)

print(f"Training data: {len(train_df):,} samples")
print(f"Test data: {len(test_df)} samples")

In [ ]:
def preprocess_text(text):
    """
    Clean tweet text — same function as all other notebooks.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Preprocess all data
print("Preprocessing training data...")
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
train_df['label'] = train_df['sentiment'].apply(lambda x: 1 if x == 4 else 0)
train_df = train_df[train_df['clean_text'].str.len() > 0]
print(f"Training size: {len(train_df):,}")

print("Preprocessing test data...")
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
test_df = test_df[test_df['clean_text'].str.len() > 0]
print(f"Test size: {len(test_df)}")

In [ ]:
# Sample for grid search (full 1.6M would take too long)
SAMPLE_SIZE = 200000

np.random.seed(42)
sample_df = train_df.sample(n=SAMPLE_SIZE, random_state=42)

X_sample = sample_df['clean_text'].values
y_sample = sample_df['label'].values

print(f"Grid search sample: {SAMPLE_SIZE:,} tweets")
print(f"  Negative: {(y_sample == 0).sum():,}")
print(f"  Positive: {(y_sample == 1).sum():,}")

---
## 2. Grid Search

We manually run the grid search (instead of sklearn's `GridSearchCV`) because we need to tune **both** the TF-IDF vectorizer and the classifier together — `GridSearchCV` with a `Pipeline` would work too, but this approach gives us more control and visibility.

In [ ]:
# Define search space
param_grid = {
    'max_features': [10000, 30000, 50000],
    'ngram_range': [(1, 1), (1, 2), (1, 3)],
    'sublinear_tf': [False, True],
    'C': [0.01, 0.1, 1.0, 10.0]
}

total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)

print(f"Total configurations to test: {total_combos}")
print(f"Each with 3-fold CV on {SAMPLE_SIZE:,} samples")
print(f"\nThis will take a while — estimated 30-60 minutes.")

In [ ]:
# Run grid search
results = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

combos = list(product(
    param_grid['max_features'],
    param_grid['ngram_range'],
    param_grid['sublinear_tf'],
    param_grid['C']
))

print(f"Running {len(combos)} configurations...\n")
overall_start = time.time()

for i, (max_feat, ngram, sublinear, C) in enumerate(combos):
    start = time.time()
    
    # Create vectorizer with these params
    tfidf = TfidfVectorizer(
        max_features=max_feat,
        ngram_range=ngram,
        min_df=2,
        max_df=0.95,
        sublinear_tf=sublinear
    )
    
    # Vectorize
    X_tfidf = tfidf.fit_transform(X_sample)
    
    # Create model
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    
    # Cross-validate
    scores = cross_val_score(model, X_tfidf, y_sample, cv=cv, scoring='accuracy')
    
    elapsed = time.time() - start
    
    result = {
        'max_features': max_feat,
        'ngram_range': str(ngram),
        'sublinear_tf': sublinear,
        'C': C,
        'mean_cv_accuracy': scores.mean(),
        'std_cv_accuracy': scores.std(),
        'time_seconds': elapsed
    }
    results.append(result)
    
    # Progress update every 10 configs
    if (i + 1) % 10 == 0 or i == 0:
        total_elapsed = time.time() - overall_start
        est_remaining = (total_elapsed / (i + 1)) * (len(combos) - i - 1)
        print(f"[{i+1}/{len(combos)}] max_feat={max_feat}, ngram={ngram}, "
              f"sublinear={sublinear}, C={C} → {scores.mean():.4f} "
              f"(~{est_remaining/60:.0f} min remaining)")

total_time = time.time() - overall_start
print(f"\nGrid search complete! Total time: {total_time/60:.1f} minutes")

---
## 3. Results Analysis

In [ ]:
# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mean_cv_accuracy', ascending=False).reset_index(drop=True)

print("Top 10 Configurations:")
print("=" * 90)
print(results_df[['max_features', 'ngram_range', 'sublinear_tf', 'C', 
                   'mean_cv_accuracy', 'std_cv_accuracy']].head(10).to_string(index=False))

In [ ]:
# Bottom 10 (worst configs)
print("\nBottom 5 Configurations:")
print("=" * 90)
print(results_df[['max_features', 'ngram_range', 'sublinear_tf', 'C', 
                   'mean_cv_accuracy', 'std_cv_accuracy']].tail(5).to_string(index=False))

In [ ]:
# Compare current (baseline) vs best
baseline = results_df[
    (results_df['max_features'] == 10000) &
    (results_df['ngram_range'] == '(1, 2)') &
    (results_df['sublinear_tf'] == False) &
    (results_df['C'] == 1.0)
].iloc[0]

best = results_df.iloc[0]

print("Baseline (current) vs Best Configuration:")
print("=" * 60)
print(f"{'Parameter':<20} {'Baseline':>15} {'Best':>15}")
print("-" * 60)
print(f"{'max_features':<20} {baseline['max_features']:>15,} {best['max_features']:>15,}")
print(f"{'ngram_range':<20} {baseline['ngram_range']:>15} {best['ngram_range']:>15}")
print(f"{'sublinear_tf':<20} {str(baseline['sublinear_tf']):>15} {str(best['sublinear_tf']):>15}")
print(f"{'C':<20} {baseline['C']:>15} {best['C']:>15}")
print("-" * 60)
print(f"{'CV Accuracy':<20} {baseline['mean_cv_accuracy']:>14.4f}% {best['mean_cv_accuracy']:>14.4f}%")
print(f"{'Improvement':<20} {'':>15} {best['mean_cv_accuracy'] - baseline['mean_cv_accuracy']:>+14.4f}%")

In [ ]:
# Analyze effect of each parameter
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Effect of max_features
param_effect = results_df.groupby('max_features')['mean_cv_accuracy'].mean()
axes[0, 0].bar(param_effect.index.astype(str), param_effect.values, color='#3498db')
axes[0, 0].set_title('Effect of max_features')
axes[0, 0].set_ylabel('Mean CV Accuracy')
axes[0, 0].set_ylim(param_effect.min() - 0.005, param_effect.max() + 0.005)
for i, (idx, v) in enumerate(param_effect.items()):
    axes[0, 0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

# 2. Effect of ngram_range
param_effect = results_df.groupby('ngram_range')['mean_cv_accuracy'].mean()
axes[0, 1].bar(param_effect.index, param_effect.values, color='#e67e22')
axes[0, 1].set_title('Effect of ngram_range')
axes[0, 1].set_ylabel('Mean CV Accuracy')
axes[0, 1].set_ylim(param_effect.min() - 0.005, param_effect.max() + 0.005)
for i, (idx, v) in enumerate(param_effect.items()):
    axes[0, 1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

# 3. Effect of sublinear_tf
param_effect = results_df.groupby('sublinear_tf')['mean_cv_accuracy'].mean()
axes[1, 0].bar(['False', 'True'], param_effect.values, color='#2ecc71')
axes[1, 0].set_title('Effect of sublinear_tf')
axes[1, 0].set_ylabel('Mean CV Accuracy')
axes[1, 0].set_ylim(param_effect.min() - 0.005, param_effect.max() + 0.005)
for i, v in enumerate(param_effect.values):
    axes[1, 0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

# 4. Effect of C
param_effect = results_df.groupby('C')['mean_cv_accuracy'].mean()
axes[1, 1].bar(param_effect.index.astype(str), param_effect.values, color='#8e44ad')
axes[1, 1].set_title('Effect of C (regularization)')
axes[1, 1].set_ylabel('Mean CV Accuracy')
axes[1, 1].set_ylim(param_effect.min() - 0.005, param_effect.max() + 0.005)
for i, (idx, v) in enumerate(param_effect.items()):
    axes[1, 1].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('Effect of Each Hyperparameter on CV Accuracy\n(averaged over all other parameters)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: ngram_range vs max_features (averaged over C and sublinear_tf)
pivot = results_df.pivot_table(
    values='mean_cv_accuracy', 
    index='ngram_range', 
    columns='max_features', 
    aggfunc='mean'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', linewidths=1)
plt.title('CV Accuracy: ngram_range vs max_features\n(averaged over C and sublinear_tf)')
plt.ylabel('ngram_range')
plt.xlabel('max_features')
plt.tight_layout()
plt.show()

---
## 4. Best Model Evaluation

Train the best configuration on the full sample and evaluate on the held-out test set.

In [ ]:
# Extract best params
best_params = results_df.iloc[0]
print("Best configuration:")
print(f"  max_features: {best_params['max_features']}")
print(f"  ngram_range:  {best_params['ngram_range']}")
print(f"  sublinear_tf: {best_params['sublinear_tf']}")
print(f"  C:            {best_params['C']}")
print(f"  CV Accuracy:  {best_params['mean_cv_accuracy']:.4f} ± {best_params['std_cv_accuracy']:.4f}")

In [ ]:
# Parse ngram_range string back to tuple
best_ngram = eval(best_params['ngram_range'])

# Train best model on full sample
print("Training best model on 200k sample...")
best_tfidf = TfidfVectorizer(
    max_features=int(best_params['max_features']),
    ngram_range=best_ngram,
    min_df=2,
    max_df=0.95,
    sublinear_tf=bool(best_params['sublinear_tf'])
)

X_train_best = best_tfidf.fit_transform(X_sample)
best_model = LogisticRegression(C=float(best_params['C']), max_iter=1000, random_state=42)
best_model.fit(X_train_best, y_sample)
print("Done!")

# Also train baseline for comparison
print("\nTraining baseline model for comparison...")
baseline_tfidf = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95
)
X_train_baseline = baseline_tfidf.fit_transform(X_sample)
baseline_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
baseline_model.fit(X_train_baseline, y_sample)
print("Done!")

In [ ]:
# Evaluate both on test set
POSITIVE_THRESHOLD = 0.6
NEGATIVE_THRESHOLD = 0.4

def predict_sentiment(probability):
    if probability > POSITIVE_THRESHOLD:
        return 'positive'
    elif probability < NEGATIVE_THRESHOLD:
        return 'negative'
    else:
        return 'neutral'

label_map = {0: 'negative', 2: 'neutral', 4: 'positive'}
y_true = test_df['sentiment'].map(label_map).values

# Best model predictions
X_test_best = best_tfidf.transform(test_df['clean_text'])
y_proba_best = best_model.predict_proba(X_test_best)[:, 1]
y_pred_best = [predict_sentiment(p) for p in y_proba_best]

# Baseline predictions
X_test_baseline = baseline_tfidf.transform(test_df['clean_text'])
y_proba_baseline = baseline_model.predict_proba(X_test_baseline)[:, 1]
y_pred_baseline = [predict_sentiment(p) for p in y_proba_baseline]

# 3-class accuracy
acc_best_3 = accuracy_score(y_true, y_pred_best)
acc_base_3 = accuracy_score(y_true, y_pred_baseline)

# Binary accuracy
binary_mask = test_df['sentiment'] != 2
acc_best_bin = accuracy_score(
    test_df.loc[binary_mask, 'sentiment'].map(label_map),
    ['positive' if p >= 0.5 else 'negative' for p in y_proba_best[binary_mask]]
)
acc_base_bin = accuracy_score(
    test_df.loc[binary_mask, 'sentiment'].map(label_map),
    ['positive' if p >= 0.5 else 'negative' for p in y_proba_baseline[binary_mask]]
)

print("Test Set Results: Baseline vs Tuned")
print("=" * 55)
print(f"{'Metric':<25} {'Baseline':>12} {'Tuned':>12}")
print("-" * 55)
print(f"{'3-class accuracy':<25} {acc_base_3:>12.2%} {acc_best_3:>12.2%}")
print(f"{'Binary accuracy':<25} {acc_base_bin:>12.2%} {acc_best_bin:>12.2%}")
print("-" * 55)
print(f"{'3-class improvement':<25} {'':>12} {acc_best_3 - acc_base_3:>+12.2%}")
print(f"{'Binary improvement':<25} {'':>12} {acc_best_bin - acc_base_bin:>+12.2%}")

In [ ]:
# Classification report for tuned model
print("Classification Report (Tuned Model):")
print(classification_report(y_true, y_pred_best, labels=['negative', 'neutral', 'positive']))

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline
cm_base = confusion_matrix(y_true, y_pred_baseline, labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
axes[0].set_title(f'Baseline (acc: {acc_base_3:.2%})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Tuned
cm_best = confusion_matrix(y_true, y_pred_best, labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'])
axes[1].set_title(f'Tuned (acc: {acc_best_3:.2%})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle('Confusion Matrix: Baseline vs Tuned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Retrain Best Config on Full Data

Now that we know the best hyperparameters, retrain on **all 1.6M samples** for the final production model.

In [ ]:
# Retrain on full data with best params
print("Retraining best configuration on full 1.6M dataset...")
print(f"  max_features: {int(best_params['max_features'])}")
print(f"  ngram_range:  {best_ngram}")
print(f"  sublinear_tf: {bool(best_params['sublinear_tf'])}")
print(f"  C:            {float(best_params['C'])}")
print()

start = time.time()

# Full TF-IDF
final_tfidf = TfidfVectorizer(
    max_features=int(best_params['max_features']),
    ngram_range=best_ngram,
    min_df=2,
    max_df=0.95,
    sublinear_tf=bool(best_params['sublinear_tf'])
)

print("Fitting TF-IDF on all training data...")
X_train_full = final_tfidf.fit_transform(train_df['clean_text'])
y_train_full = train_df['label'].values
print(f"Matrix shape: {X_train_full.shape}")

print("Training Logistic Regression...")
final_model = LogisticRegression(
    C=float(best_params['C']),
    max_iter=1000,
    random_state=42
)
final_model.fit(X_train_full, y_train_full)

elapsed = time.time() - start
print(f"\nTraining complete! ({elapsed:.1f} seconds)")

In [ ]:
# Evaluate final model on test set
X_test_final = final_tfidf.transform(test_df['clean_text'])
y_proba_final = final_model.predict_proba(X_test_final)[:, 1]
y_pred_final = [predict_sentiment(p) for p in y_proba_final]

acc_final_3 = accuracy_score(y_true, y_pred_final)
acc_final_bin = accuracy_score(
    test_df.loc[binary_mask, 'sentiment'].map(label_map),
    ['positive' if p >= 0.5 else 'negative' for p in y_proba_final[binary_mask]]
)

print("Final Model (tuned + full data) vs Baseline:")
print("=" * 60)
print(f"{'Metric':<25} {'Baseline':>12} {'Tuned+Full':>14}")
print("-" * 60)
print(f"{'Training data':<25} {'200k':>12} {'1.6M':>14}")
print(f"{'3-class accuracy':<25} {acc_base_3:>12.2%} {acc_final_3:>14.2%}")
print(f"{'Binary accuracy':<25} {acc_base_bin:>12.2%} {acc_final_bin:>14.2%}")
print("-" * 60)
print(f"{'Binary improvement':<25} {'':>12} {acc_final_bin - acc_base_bin:>+14.2%}")

print("\nClassification Report (Final Model):")
print(classification_report(y_true, y_pred_final, labels=['negative', 'neutral', 'positive']))

In [ ]:
# Test with sample sentences
test_texts = [
    "I love this product!",
    "This is terrible.",
    "The meeting is at 3pm.",
    "I can't believe how bad this is",
    "Not bad at all, actually quite good!",
]

print("Final Tuned Model Predictions:")
print("=" * 60)
for text in test_texts:
    clean = preprocess_text(text)
    vec = final_tfidf.transform([clean])
    prob = final_model.predict_proba(vec)[0][1]
    sentiment = predict_sentiment(prob)
    print(f"'{text}'")
    print(f"  -> {sentiment} (P(pos): {prob:.4f})")
    print()

In [ ]:
# Save the tuned model (commented out)
# os.makedirs('models', exist_ok=True)
# joblib.dump(final_tfidf, 'models/vectorizer_tuned.joblib')
# joblib.dump(final_model, 'models/model_tuned.joblib')
# print("Tuned model saved!")

print("Note: Model saving is commented out.")
print("Uncomment the lines above to save the tuned model.")

---
## Summary

### Grid Search Results
- Searched **72 configurations** (3 × 3 × 2 × 4) with 3-fold CV on 200k samples
- Best configuration found and retrained on full 1.6M dataset

### Key Findings

1. **max_features**: Larger vocabularies generally help — the model benefits from seeing more words
2. **ngram_range**: Bigrams improve over unigrams; trigrams may or may not help further
3. **sublinear_tf**: Log-normalization of term frequencies typically helps
4. **C (regularization)**: Moderate regularization (C=0.1 to 1.0) tends to work best

### Why Tuning Matters
- The default configuration was chosen without evidence — grid search lets us make **data-driven decisions**
- Even small accuracy gains compound when processing millions of inferences
- Shows we understand the relationship between hyperparameters and model behavior